### DSA2025 第14講：講義資料【単回帰・重回帰分析】

参考資料：[重回帰分析とは？概要から分析の流れまでわかりやすく解説](https://data-viz-lab.com/multiple-regression-analysis)

Pythonを用いた回帰には様々な方法があるが、今回はstatsmodels を用いる。データを一度、Pandasのdataframeに変換する必要がある。  
ここでは 最小二乗法を扱う ols を利用する。　詳細は例えば [こちら](https://qiita.com/innovation1005/items/b712ce54a7a697a9bf03)  
また、
+ statsmodels.api
+ statsmodels.formula.api  
の2通りあるが、ここでは下のformula付を用いる。この場合、y切片を defaultで持つので、statsmodels.api のように add_constant は不要。

In [ ]:
# 必要なライブラリのインストール
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# グラフをinline表示可能にする
%matplotlib inline

# 解像度を上げてinline表示する
%config InlineBackend.figure_format = 'retina'

# 日本語表示用ライブラリ（matplotlibで日本語を使用したい場合）
try:
    import japanize_matplotlib
except ModuleNotFoundError:
    # 初回だけインストールするので時間がかかる
    !pip install japanize-matplotlib
    import japanize_matplotlib

In [ ]:
# pairplot 作成用ライブラリ読み込み
import seaborn as sns

# ネット情報取得用ライブラリ読み込み
import urllib.request

# 回帰用ライブラリ
try:
    from statsmodels.formula.api import ols
except ModuleNotFoundError:
    # 初回だけインストールするので時間がかかる
    !pip install statsmodels
    from statsmodels.formula.api import ols

# 計量経済分析用データセット
try:
    import wooldridge
except ModuleNotFoundError:
    # 初回だけインストールするので時間がかかる
    !pip install wooldridge
    import wooldridge

#### 回帰分析（重回帰）：家賃を決める要素を説明してみる
[連載！データ分析②：回帰分析](https://www.asakonet.co.jp/asunomikata/contents/1782/20250214_watanabe-akira_2_regression/)  
【データ出典】 [データ分析学習環境「Connect DB」](https://cdb.eplang.jp/)  

In [ ]:
# excelファイルの読み込み
df_rent = pd.read_excel('https://ccs-kumanolab.com/lecture/dsa/rent_mreg.xlsx')

# 最初の5件表示
df_rent.head()

In [ ]:
# 散布図の作成
plt.figure(figsize=(4,3))
plt.scatter(df_rent['面積'], df_rent['家賃'])  # plt.scatter(shoe_size, height) でもよい

# 諸設定
plt.title("家賃と面積の関係", fontsize=14)

# 上付き文字
plt.xlabel('面積(m$^2$)', fontsize=12)
plt.ylabel('家賃(円)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 単回帰（線形回帰）
res = ols("家賃 ~ 面積", data=df_rent).fit()
res.summary()

In [ ]:
# 予測値の取得
print(res.predict())

In [ ]:
# 散布図の作成
plt.figure(figsize=(4,3))
plt.scatter(df_rent['面積'], df_rent['家賃'])
plt.plot(df_rent['面積'], res.predict(), 'r-')

# 諸設定
plt.title("家賃と面積の関係", fontsize=14)
plt.xlabel('面積(m$^2$)', fontsize=12)
plt.ylabel('家賃(円)', fontsize=12)
plt.tight_layout()
plt.show()

#### 重回帰：家賃/駅徒歩/築年数/面積/階 全ての変数間の関係をペアプロットとヒートマップで一覧表示

In [ ]:
# pairplot 作成
sns.pairplot(df_rent, height=1)
plt.show()

In [ ]:
# ヒートマップ作成
sns.heatmap(df_rent.corr(numeric_only=True), annot=True, square=True, vmax=1, vmin=-1, center=0, fmt='.2f')
plt.show()

#### 重回帰：家賃を説明するのに、面積以外の要因も考慮する

In [ ]:
df_rent.head()

In [ ]:
# 重回帰（非推奨：標準化せず）
res = ols("家賃 ~ 駅徒歩 + 築年数 + 面積 + 階", data=df_rent).fit()
res.summary()

#### 重回帰：データを標準化する
各変数の影響の強さを揃える  [回帰分析と標準化](https://analyze.kyoto/blog/%E5%9B%9E%E5%B8%B0%E5%88%86%E6%9E%90%E3%81%A8%E6%A8%99%E6%BA%96%E5%8C%96/)

In [ ]:
# 標準正規分布を作成するのと同じ手順

def z_transform(x):
    mu = x.mean()
    sigma = x.std(ddof=0)
    return (x-mu)/sigma

In [ ]:
df_rent['家賃'] = z_transform(df_rent['家賃'])
df_rent['駅徒歩'] = z_transform(df_rent['駅徒歩'])
df_rent['築年数'] = z_transform(df_rent['築年数'])
df_rent['面積'] = z_transform(df_rent['面積'])
df_rent['階'] = z_transform(df_rent['階'])
df_rent.head()

In [ ]:
# 重回帰（１次式）
res = ols("家賃 ~ 駅徒歩 + 築年数 + 面積 + 階", data=df_rent).fit()
res.summary()